In [16]:
import os
import pandas as pd
import ifcopenshell
import ifcopenshell.util.element
import ifcopenshell.api  # <-- Importante para escrever no IFC
import joblib
import json
import numpy as np
import traceback

In [17]:
# --- 1. Arquivo de Entrada ---
# Altere este caminho para o novo arquivo IFC que você quer classificar
IFC_FILE_PATH = r"C:\Users\lucas.galicioli\Downloads\RÔGGA EMPREENDIMENTOS-BRUSQUE HOME CLUB-2025-10-16-13-35-31-469\PHN21043-ARQ-EX-0002-BIM-TOR-GER-R01.ifc"

# --- 2. Artefatos do Modelo de Classificação de Elementos (Solibri) ---
MODEL_PATH = 'ifc_classifier_disciplinas_v1.pkl'
ENCODER_PATH = 'label_encoder_disciplinas_v1.pkl'
COLUMNS_PATH = 'colunas_modelo_disciplinas.json'
MEDIANS_PATH = 'medianas_treinamento.json'

# --- 3. Artefatos do Modelo de Classificação de Disciplina ---
MATRIX_PATH = r'C:\Users\lucas.galicioli\ifc-classifier\data\interim\classification-matrix.xlsx'
COLUNA_MATRIX_FILENAME = 'FileName'            # Nome da coluna de arquivos na matriz
COLUNA_MATRIX_DISCIPLINA = 'Ô_CLS_DISCIPLINAS' # Nome da coluna de disciplina na matriz

# --- 4. Configurações de Saída ---
# O nome do Pset que será criado/usado no IFC
PSET_NAME = "SOLIBRI_CLASSIFICACAO" 
# Caminho do novo arquivo IFC que será salvo
OUTPUT_IFC_PATH = os.path.join(
    os.path.dirname(IFC_FILE_PATH), 
    f"classificado_{os.path.basename(IFC_FILE_PATH)}"
)
# Caminho para o CSV de verificação (opcional, mas útil)
OUTPUT_CSV_PATH = os.path.join(
    os.path.dirname(IFC_FILE_PATH), 
    f"classificado_{os.path.splitext(os.path.basename(IFC_FILE_PATH))[0]}.csv"
)

In [18]:
def get_building_storey(element):
    """ Encontra o IfcBuildingStorey no qual o elemento está contido. """
    try:
        spatial_container = ifcopenshell.util.element.get_container(element)
        if spatial_container and spatial_container.is_a('IfcBuildingStorey'):
            return spatial_container.Name
    except Exception:
        pass
    return None

def get_material_name(element):
    """ Extrai o nome do material associado ao elemento. """
    material = ifcopenshell.util.element.get_material(element)
    if not material: return None
    if hasattr(material, 'Name'): return material.Name
    elif hasattr(material, 'MaterialLayers'):
        layer_names = [
            layer.Material.Name for layer in material.MaterialLayers
            if hasattr(layer, 'Material') and hasattr(layer.Material, 'Name')
        ]
        return ', '.join(layer_names) if layer_names else None
    return None

def get_quantity_value_legacy(element, quantity_name):
    """ Busca por uma quantidade específica (ex: 'Width') e retorna seu valor. """
    for definition in getattr(element, 'IsDefinedBy', []):
        if definition.is_a('IfcRelDefinesByProperties'):
            prop_set = definition.RelatingPropertyDefinition
            if prop_set.is_a('IfcElementQuantity'):
                for quantity in prop_set.Quantities:
                    if quantity.Name == quantity_name:
                        value_attribute = next((attr for attr in dir(quantity) if attr.endswith('Value')), None)
                        if value_attribute:
                            return getattr(quantity, value_attribute)
    return None

def gerar_mapa_disciplinas(matrix_path, col_filename, col_disciplina):
    """
    Carrega a matriz de classificação e cria um dicionário de mapeamento
    (Código -> Disciplina). Ex: {'HID': 'Hidrossanitário', 'EST': 'Estrutura'}
    """
    try:
        df_matrix = pd.read_excel(matrix_path)
        df_mapa_temp = df_matrix[[col_filename, col_disciplina]].copy()
        df_mapa_temp['Disciplina_Code'] = df_mapa_temp[col_filename].str.split('-').str[1]
        df_mapa_temp = df_mapa_temp[['Disciplina_Code', col_disciplina]].dropna().drop_duplicates()
        mapa = df_mapa_temp.set_index('Disciplina_Code')[col_disciplina].to_dict()
        
        if not mapa:
            print(f"AVISO: O mapa de disciplinas gerado a partir de '{matrix_path}' está vazio.")
        return mapa
    except FileNotFoundError:
        print(f"ERRO: Arquivo da matriz não encontrado em: {matrix_path}")
    except KeyError as e:
        print(f"ERRO: Coluna {e} não encontrada na matriz. Verifique os nomes '{col_filename}' e '{col_disciplina}'.")
    except Exception as e:
        print(f"ERRO ao gerar mapa de disciplinas: {e}")
    return None

def extrair_disciplina_do_nome(nome_arquivo_base, mapa_disciplinas):
    """ Extrai o código do nome do arquivo e o traduz usando o mapa. """
    try:
        nome_base = os.path.splitext(nome_arquivo_base)[0]
        partes_nome = nome_base.split('-')
        codigo_disciplina = partes_nome[1] # Pega o "HID", "EST", etc.
        
        disciplina_traduzida = mapa_disciplinas.get(codigo_disciplina, f"Código '{codigo_disciplina}' Não Mapeado")
        return disciplina_traduzida
    except IndexError:
        print(f"AVISO: O nome '{nome_arquivo_base}' não segue o padrão 'XXX-CODIGO-...'.")
        return "Erro: Padrão de Nome"
    except Exception:
        return "Erro: Extração"

In [19]:
def processar_ifc_completo():
    try:
        # --- ETAPA 1: Carregar todos os Artefatos ---
        print("Carregando artefatos do modelo treinado...")
        modelo = joblib.load(MODEL_PATH)
        label_encoder = joblib.load(ENCODER_PATH)
        with open(COLUMNS_PATH, 'r', encoding='utf-8') as f:
            colunas_do_modelo = json.load(f)
        with open(MEDIANS_PATH, 'r', encoding='utf-8') as f:
            medianas_treinamento = json.load(f)
        
        print("Carregando mapa de disciplinas...")
        mapa_disciplinas = gerar_mapa_disciplinas(MATRIX_PATH, COLUNA_MATRIX_FILENAME, COLUNA_MATRIX_DISCIPLINA)
        if not mapa_disciplinas:
            print("AVISO CRÍTICO: Mapa de disciplinas não carregado. A classificação de disciplina falhará.")
            # Você pode decidir parar aqui se a disciplina for essencial
            # return

        print("-> Artefatos carregados com sucesso!")

        # --- ETAPA 2: Extrair dados do IFC ---
        print(f"\nIniciando processamento do arquivo IFC: {os.path.basename(IFC_FILE_PATH)}...")
        ifc_file = ifcopenshell.open(IFC_FILE_PATH)
        file_name = os.path.basename(IFC_FILE_PATH)
        products = ifc_file.by_type('IfcProduct')
        element_data = []

        for product in products:
            if product.is_a('IfcOpeningElement') or product.is_a('IfcVirtualElement'):
                continue

            psets = ifcopenshell.util.element.get_psets(product)
            rogga_pset = psets.get('PSET_RÔGGA', {})

            element_info = {
                'GlobalId': product.GlobalId,
                'Class': product.is_a(),
                'PredefinedType': getattr(product, 'PredefinedType', None),
                'Name': getattr(product, 'Name', None),
                'BuildingStorey': get_building_storey(product),
                'Material': get_material_name(product),
                'PSET_RÔGGA.RÔGGA_SEÇÃO': rogga_pset.get('RÔGGA_SEÇÃO', None),
                'PSET_RÔGGA.RÔGGA_DESCRIÇÃO': rogga_pset.get('RÔGGA_DESCRIÇÃO', None),
                'Width': get_quantity_value_legacy(product, 'Width'),
                'Thickness': get_quantity_value_legacy(product, 'Thickness'),
                'Length': get_quantity_value_legacy(product, 'Length'),
                'Height': get_quantity_value_legacy(product, 'Height'),
                'FileName': file_name
            }
            element_data.append(element_info)

        df_ifc_data = pd.DataFrame(element_data)
        print(f"-> {len(df_ifc_data)} elementos extraídos do IFC.")

        if df_ifc_data.empty:
            print("AVISO: Nenhum elemento válido foi extraído do IFC. Encerrando.")
            return

        # --- ETAPA 3: Classificação de Elementos (ML) ---
        print("\nIniciando pré-processamento e classificação de ELEMENTOS...")
        features_selecionadas = [
            'Class', 'PredefinedType', 'BuildingStorey', 'Material',
            'PSET_RÔGGA.RÔGGA_SEÇÃO', 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO',
            'Width', 'Thickness', 'Length', 'Height'
        ]
        df_para_prever = df_ifc_data[features_selecionadas].copy()

        cat_cols = df_para_prever.select_dtypes(include=['object']).columns
        df_para_prever.loc[:, cat_cols] = df_para_prever.loc[:, cat_cols].fillna('Desconhecido')
        for col, mediana in medianas_treinamento.items():
            if col in df_para_prever.columns:
                 df_para_prever.loc[:, col] = df_para_prever.loc[:, col].fillna(mediana)

        df_encodado = pd.get_dummies(df_para_prever)
        df_encodado.columns = df_encodado.columns.str.replace(r'\\[|\\]|<', '_', regex=True)
        df_final = df_encodado.reindex(columns=colunas_do_modelo, fill_value=0)

        previsoes_numericas = modelo.predict(df_final)
        previsoes_texto_solibri = label_encoder.inverse_transform(previsoes_numericas)
        df_ifc_data['Ô_CLS_CLASSIFICAÇÃO_SOLIBRI'] = previsoes_texto_solibri
        print("-> Classificação de elementos (ML) concluída.")

        # --- ETAPA 4: Classificação de Disciplina (Mapa) ---
        print("\nIniciando classificação de DISCIPLINA...")
        disciplina_final = "Erro ao Mapear"
        if mapa_disciplinas:
            disciplina_final = extrair_disciplina_do_nome(file_name, mapa_disciplinas)
        
        df_ifc_data['Ô_CLS_DISCIPLINAS'] = disciplina_final
        print(f"-> Classificação de disciplina ('{disciplina_final}') concluída.")

        # --- ETAPA 5: Preparar Mapa de Escrita ---
        # Criamos um dicionário onde a chave é o GlobalId e o valor é um dicionário com as propriedades
        df_mapa_final = df_ifc_data[['GlobalId', 'Ô_CLS_CLASSIFICAÇÃO_SOLIBRI', 'Ô_CLS_DISCIPLINAS']]
        mapa_de_escrita = df_mapa_final.set_index('GlobalId').to_dict('index')
        
        # Ex: {'3zo9X...V': {'Ô_CLS_CLASSIFICAÇÃO_SOLIBRI': 'Bombas...', 'Ô_CLS_DISCIPLINAS': 'Arquitetura'}}

        print(f"\n--- AMOSTRA DO RESULTADO COMBINADO (em memória) ---")
        print(df_ifc_data[['GlobalId', 'Name', 'Ô_CLS_CLASSIFICAÇÃO_SOLIBRI', 'Ô_CLS_DISCIPLINAS']].head())

        # --- ETAPA 6: Escrever dados de volta no IFC ---
        print(f"\nIniciando gravação das propriedades no Pset '{PSET_NAME}' do arquivo IFC...")
        
        elementos_modificados = 0
        for global_id, propriedades in mapa_de_escrita.items():
            elemento = ifc_file.by_guid(global_id)
            if not elemento:
                print(f"AVISO: Elemento com GlobalId {global_id} não encontrado no ifc_file. Pulando.")
                continue
            
            try:
                # 1. Criar o Pset
                pset = ifcopenshell.api.run("pset.add_pset", 
                                            ifc_file, 
                                            product=elemento, 
                                            name=PSET_NAME)
                
                # 2. Editar o Pset com as novas propriedades
                ifcopenshell.api.run("pset.edit_pset", 
                                     ifc_file, 
                                     pset=pset, 
                                     properties=propriedades)
                elementos_modificados += 1
            except Exception as e:
                print(f"ERRO ao tentar gravar Pset no elemento {global_id}: {e}")

        print(f"-> Propriedades gravadas com sucesso em {elementos_modificados} elementos.")

        # --- ETAPA 7: Salvar os arquivos de saída (IFC e CSV) ---
        
        # 7.1. Salvar o NOVO arquivo IFC modificado
        ifc_file.write(OUTPUT_IFC_PATH)
        print(f"\nArquivo IFC modificado salvo com sucesso em: {os.path.abspath(OUTPUT_IFC_PATH)}")

        # 7.2. Salvar o CSV de verificação
        df_ifc_data.to_csv(OUTPUT_CSV_PATH, index=False, sep=';', decimal=',', encoding='utf-8-sig')
        print(f"Arquivo CSV de verificação salvo em: {os.path.abspath(OUTPUT_CSV_PATH)}")

    except FileNotFoundError as e:
        print(f"ERRO CRÍTICO: Arquivo não encontrado. {e}")
        traceback.print_exc()
    except Exception as e:
        print(f"Ocorreu um erro inesperado durante o processamento: {e}")
        traceback.print_exc()

# --- Executa a função principal ---
processar_ifc_completo()

Carregando artefatos do modelo treinado...
Carregando mapa de disciplinas...
-> Artefatos carregados com sucesso!

Iniciando processamento do arquivo IFC: PHN21043-ARQ-EX-0002-BIM-TOR-GER-R01.ifc...
-> 42052 elementos extraídos do IFC.

Iniciando pré-processamento e classificação de ELEMENTOS...
-> Classificação de elementos (ML) concluída.

Iniciando classificação de DISCIPLINA...
-> Classificação de disciplina ('Arquitetura') concluída.

--- AMOSTRA DO RESULTADO COMBINADO (em memória) ---
                 GlobalId                                     Name  \
0  3PbWJw7mv2_fn0$fobZxeg  Verga e contraverga:400 x 800mm:3990740   
1  3PbWJw7mv2_fn0$fobZXQz  Verga e contraverga:400 x 800mm:4016195   
2  3PbWJw7mv2_fn0$fobZXQr  Verga e contraverga:400 x 800mm:4016203   
3  3PbWJw7mv2_fn0$fobZXQj  Verga e contraverga:400 x 800mm:4016211   
4  3PbWJw7mv2_fn0$fobZXQZ  Verga e contraverga:400 x 800mm:4016221   

    Ô_CLS_CLASSIFICAÇÃO_SOLIBRI Ô_CLS_DISCIPLINAS  
0  Sanca, cortineiro e testeira

In [20]:
# Verifique se o caminho de saída (OUTPUT_IFC_PATH) da Célula 2 está correto
if 'OUTPUT_IFC_PATH' in locals() and os.path.exists(OUTPUT_IFC_PATH):
    print(f"Verificando o arquivo salvo: {OUTPUT_IFC_PATH}")
    
    try:
        ifc_verificacao = ifcopenshell.open(OUTPUT_IFC_PATH)
        
        # Pega o primeiro IfcProduct que não seja Abertura/Virtual
        primeiro_produto = None
        for p in ifc_verificacao.by_type('IfcProduct'):
             if not (p.is_a('IfcOpeningElement') or p.is_a('IfcVirtualElement')):
                primeiro_produto = p
                break
        
        if primeiro_produto:
            print(f"\n--- Verificando Psets do elemento: {primeiro_produto.Name} (ID: {primeiro_produto.id()}) ---")
            
            psets = ifcopenshell.util.element.get_psets(primeiro_produto, psets_only=True)
            
            if PSET_NAME in psets:
                print(f"\nSUCESSO! Pset '{PSET_NAME}' encontrado:")
                print(json.dumps(psets[PSET_NAME], indent=2, ensure_ascii=False))
            else:
                print(f"\nFALHA! Pset '{PSET_NAME}' NÃO foi encontrado no elemento de amostra.")
                print("Psets disponíveis:", list(psets.keys()))
        else:
            print("Não foi possível encontrar um elemento de amostra no IFC salvo.")
            
    except Exception as e:
        print(f"Ocorreu um erro ao tentar verificar o arquivo IFC salvo: {e}")
else:
    print("Execute a Célula 4 primeiro para gerar o arquivo de saída.")

Verificando o arquivo salvo: C:\Users\lucas.galicioli\Downloads\RÔGGA EMPREENDIMENTOS-BRUSQUE HOME CLUB-2025-10-16-13-35-31-469\classificado_PHN21043-ARQ-EX-0002-BIM-TOR-GER-R01.ifc

--- Verificando Psets do elemento: Verga e contraverga:400 x 800mm:3990740 (ID: 524818) ---

SUCESSO! Pset 'SOLIBRI_CLASSIFICACAO' encontrado:
{
  "Ô_CLS_CLASSIFICAÇÃO_SOLIBRI": "Sanca, cortineiro e testeira",
  "Ô_CLS_DISCIPLINAS": "Arquitetura",
  "id": 1360681
}


In [21]:
df = pd.read_csv(OUTPUT_CSV_PATH,sep=";"
                 )
display(df)

,GlobalId,Class,PredefinedType,Name,BuildingStorey,Material,PSET_RÔGGA.RÔGGA_SEÇÃO,PSET_RÔGGA.RÔGGA_DESCRIÇÃO,Width,Thickness,Length,Height,FileName,Ô_CLS_CLASSIFICAÇÃO_SOLIBRI,Ô_CLS_DISCIPLINAS
0,3PbWJw7mv2_fn0$fobZxeg,IfcBeam,LINTEL,Verga e contraverga:400 x 800mm:3990740,35. TP1,"Concrete, Cast-in-Place gray",NaN,RÔG_VER_Verga e contraverga: 400 x 800mm,NaN,NaN,"0,8149999999999994",NaN,PHN21043-ARQ-EX-0002-BIM-TOR-GER-R01.ifc,"Sanca, cortineiro e testeira",Arquitetura
1,3PbWJw7mv2_fn0$fobZXQz,IfcBeam,LINTEL,Verga e contraverga:400 x 800mm:4016195,36. TAP,"Concrete, Cast-in-Place gray",NaN,RÔG_VER_Verga e contraverga: 400 x 800mm,NaN,NaN,"0,8149999999999994",NaN,PHN21043-ARQ-EX-0002-BIM-TOR-GER-R01.ifc,"Sanca, cortineiro e testeira",Arquitetura
2,3PbWJw7mv2_fn0$fobZXQr,IfcBeam,LINTEL,Verga e contraverga:400 x 800mm:4016203,37. TP1,"Concrete, Cast-in-Place gray",NaN,RÔG_VER_Verga e contraverga: 400 x 800mm,NaN,NaN,"0,8149999999999994",NaN,PHN21043-ARQ-EX-0002-BIM-TOR-GER-R01.ifc,"Sanca, cortineiro e testeira",Arquitetura
3,3PbWJw7mv2_fn0$fobZXQj,IfcBeam,LINTEL,Verga e contraverga:400 x 800mm:4016211,ATC,"Concrete, Cast-in-Place gray",NaN,RÔG_VER_Verga e contraverga: 400 x 800mm,NaN,NaN,"0,8149999999999994",NaN,PHN21043-ARQ-EX-0002-BIM-TOR-GER-R01.ifc,"Sanca, cortineiro e testeira",Arquitetura
4,3PbWJw7mv2_fn0$fobZXQZ,IfcBeam,LINTEL,Verga e contraverga:400 x 800mm:4016221,10. TP1,"Concrete, Cast-in-Place gray",NaN,RÔG_VER_Verga e contraverga: 400 x 800mm,NaN,NaN,"0,8149999999999994",NaN,PHN21043-ARQ-EX-0002-BIM-TOR-GER-R01.ifc,"Sanca, cortineiro e testeira",Arquitetura
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42047,0Qfacpun97Jf3FH$5seuBQ,IfcSpace,NOTDEFINED,855,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,4384",PHN21043-ARQ-EX-0002-BIM-TOR-GER-R01.ifc,"Bombas de recalque, de drenagem pluvial",Arquitetura
42048,0Qfacpun97Jf3FH$5seuBj,IfcSpace,NOTDEFINED,856,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,67",PHN21043-ARQ-EX-0002-BIM-TOR-GER-R01.ifc,"Bombas de recalque, de drenagem pluvial",Arquitetura
42049,0Qfacpun97Jf3FH$5seuBp,IfcSpace,NOTDEFINED,857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,67",PHN21043-ARQ-EX-0002-BIM-TOR-GER-R01.ifc,"Bombas de recalque, de drenagem pluvial",Arquitetura
42050,3azJv7iZT01w_YxCJ5IJTC,IfcSpace,NOTDEFINED,858,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,4384",PHN21043-ARQ-EX-0002-BIM-TOR-GER-R01.ifc,"Bombas de recalque, de drenagem pluvial",Arquitetura
